PHASE 6A — FROZEN PUBLICATION VALIDATION

Primary hypothesis:
At equal intervention budget k=2, the interaction-aware circuit
[(10,Q),(8,S)] achieves greater normalized causal recovery than
the first-order AIE circuit [(10,Q),(9,Q)] on unseen CounterFact facts.

Primary endpoint:
Mean paired difference in normalized recovery:
    delta = recovery_interaction - recovery_AIE

Primary success criterion:
    bootstrap 95% CI lower bound > 0

Practical-effect criterion:
    mean delta >= 0.02

Primary corruption:
    subject embedding Gaussian corruption
    noise multiplier = 3.0

Robustness conditions:
    noise multipliers = 2.5, 3.0, 3.5
    3 fixed noise seeds

Fresh facts:
    N = 128
    all Phase-5 facts excluded

No site selection, threshold tuning, or hyperparameter changes
after viewing Phase-6 results.

In [1]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

In [2]:
MODEL_ID = "openai-community/gpt2"

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("device:", DEVICE)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID
).to(DEVICE).eval()

N_LAYERS = model.config.n_layer

EMBED_STD = (
    model.transformer.wte.weight
    .detach()
    .std()
    .item()
)

print("layers:", N_LAYERS)
print("embedding std:", EMBED_STD)

device: cpu


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

layers: 12
embedding std: 0.14369554817676544


In [3]:
counterfact = load_dataset(
    "azhx/counterfact",
    split="test"
)

print(counterfact)

Dataset({
    features: ['case_id', 'pararel_idx', 'requested_rewrite', 'paraphrase_prompts', 'neighborhood_prompts', 'attribute_prompts', 'generation_prompts'],
    num_rows: 2191
})


In [6]:
AIE_PAIR = [
    (10, "Q"),
    (9,  "Q"),
]

INTERACTION_PAIR = [
    (10, "Q"),
    (8,  "S"),
]

PRIMARY_SCALE = 3.0

NOISE_SCALES = [
    2.5,
    3.0,
    3.5,
]

NOISE_SEEDS = [
    101,
    202,
    303,
]

N_FRESH = 128

REFERENCE_SEED = 424242
REFERENCE_SCALE = 3.0

MIN_CLEAN_MARGIN = 1.0
MIN_REFERENCE_TE = 1.0

# Per le valutazioni secondarie:
# evitiamo di dividere per TE quasi nullo.
MIN_EVAL_TE = 0.5

In [7]:
old_train = pd.read_csv(
    "phase5_mobius_train_raw.csv"
)

old_test = pd.read_csv(
    "phase5_mobius_test_raw.csv"
)

OLD_CASE_IDS = set(
    old_train["case_id"].unique()
).union(
    set(
        old_test["case_id"].unique()
    )
)

print(
    "Phase-5 excluded cases:",
    len(OLD_CASE_IDS)
)

Phase-5 excluded cases: 64


In [8]:
def one_token_id(text):

    ids = tokenizer.encode(
        " " + text.strip(),
        add_special_tokens=False
    )

    if len(ids) != 1:
        return None

    return ids[0]

In [9]:
def subject_token_positions(
    prompt,
    subject
):

    enc = tokenizer(
        prompt,
        return_offsets_mapping=True,
        add_special_tokens=False
    )

    start = prompt.find(subject)

    if start < 0:
        return []

    end = start + len(subject)

    positions = []

    for i, (a, b) in enumerate(
        enc["offset_mapping"]
    ):

        if b > start and a < end:
            positions.append(i)

    return positions

In [10]:
def logit_margin(
    logits,
    true_id,
    alt_id
):

    return (
        logits[0, -1, true_id]
        -
        logits[0, -1, alt_id]
    ).item()

In [11]:
def corrupted_embeddings(
    input_ids,
    subj_positions,
    seed,
    noise_multiplier
):

    embeds = (
        model.transformer.wte(
            input_ids
        )
        .detach()
        .clone()
    )

    g = torch.Generator(
        device=embeds.device
    )

    g.manual_seed(
        int(seed)
    )

    noise = torch.randn(
        embeds.shape,
        generator=g,
        device=embeds.device,
        dtype=embeds.dtype
    )

    noise *= (
        noise_multiplier
        * EMBED_STD
    )

    for pos in subj_positions:

        embeds[:, pos, :] += (
            noise[:, pos, :]
        )

    return embeds

In [12]:
def prepare_fresh_case(
    record,
    dataset_index
):

    rr = record[
        "requested_rewrite"
    ]

    subject = rr["subject"]

    prompt = rr["prompt"].format(
        subject
    )

    true_text = (
        rr["target_true"]["str"]
    )

    new_text = (
        rr["target_new"]["str"]
    )

    true_id = one_token_id(
        true_text
    )

    new_id = one_token_id(
        new_text
    )

    if true_id is None:
        return None

    if new_id is None:
        return None

    if true_id == new_id:
        return None

    subj_positions = (
        subject_token_positions(
            prompt,
            subject
        )
    )

    if len(subj_positions) == 0:
        return None

    enc = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    )

    input_ids = (
        enc["input_ids"]
        .to(DEVICE)
    )

    attention_mask = (
        enc["attention_mask"]
        .to(DEVICE)
    )

    with torch.no_grad():

        clean_out = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    D_clean = logit_margin(
        clean_out.logits,
        true_id,
        new_id
    )

    if D_clean < MIN_CLEAN_MARGIN:
        return None

    ref_emb = (
        corrupted_embeddings(
            input_ids,
            subj_positions,
            REFERENCE_SEED
            + int(dataset_index),
            REFERENCE_SCALE
        )
    )

    with torch.no_grad():

        ref_out = model(
            inputs_embeds=ref_emb,
            attention_mask=attention_mask
        )

    D_ref = logit_margin(
        ref_out.logits,
        true_id,
        new_id
    )

    TE_ref = (
        D_clean - D_ref
    )

    if TE_ref < MIN_REFERENCE_TE:
        return None

    return {
        "case_id":
            int(record["case_id"]),

        "dataset_index":
            int(dataset_index),

        "prompt":
            prompt,

        "subject":
            subject,

        "true_text":
            true_text,

        "new_text":
            new_text,

        "true_id":
            int(true_id),

        "new_id":
            int(new_id),

        "input_ids":
            input_ids,

        "attention_mask":
            attention_mask,

        "subject_positions":
            subj_positions,

        "subject_last":
            subj_positions[-1],

        "query_pos":
            input_ids.shape[1] - 1,

        "D_clean":
            float(D_clean),

        "TE_reference":
            float(TE_ref)
    }

In [13]:
rng = np.random.default_rng(
    20260909
)

indices = rng.permutation(
    len(counterfact)
)

fresh_cases = []

for idx in indices:

    record = counterfact[
        int(idx)
    ]

    case_id = int(
        record["case_id"]
    )

    if case_id in OLD_CASE_IDS:
        continue

    case = prepare_fresh_case(
        record,
        int(idx)
    )

    if case is None:
        continue

    fresh_cases.append(case)

    print(
        len(fresh_cases),
        case["subject"],
        "| reference TE =",
        round(
            case["TE_reference"],
            3
        )
    )

    if len(fresh_cases) >= N_FRESH:
        break

1 thorium | reference TE = 3.971
2 Co-operatives UK | reference TE = 3.127
3 Uruguayan Football Association | reference TE = 9.509
4 Ferrari 250 GTO | reference TE = 3.758
5 Kevin Garnett | reference TE = 1.642
6 Mauritius | reference TE = 1.28
7 Il giuramento | reference TE = 1.302
8 Victoria Derbyshire | reference TE = 1.086
9 Orpheus Island National Park | reference TE = 1.254
10 Il Posto | reference TE = 1.476
11 Aleksander Griboyedov | reference TE = 3.46
12 Everberg | reference TE = 2.983
13 Ghora Gali | reference TE = 2.25
14 Honda CRF series | reference TE = 7.697
15 Andrei Bitov | reference TE = 1.342
16 2004 Australian Open | reference TE = 4.83
17 Ludovic Sylvestre | reference TE = 1.943
18 Korean Wikipedia | reference TE = 6.738
19 Babylas of Antioch | reference TE = 1.098
20 North Dakota | reference TE = 1.328
21 Chicago Rockets | reference TE = 7.232
22 Veronica Lario | reference TE = 1.748
23 Pulaski County Special School District | reference TE = 3.588
24 Great Soviet E

In [34]:
# Nuovo fix senza overlap
phase5_seen = pd.read_csv(
    "phase5_seen_facts.csv"
)

OLD_SUBJECTS = set(
    phase5_seen["subject_key"]
)

print(
    "excluded Phase-5 subjects:",
    len(OLD_SUBJECTS)
)

excluded Phase-5 subjects: 64


In [35]:
phase5_seen = pd.read_csv(
    "phase5_seen_facts.csv"
)

phase6_v1_manifest = pd.read_csv(
    "phase6_fresh_manifest.csv"
)

PHASE5_SUBJECTS = set(
    phase5_seen["subject"]
    .str.strip()
    .str.lower()
)

PHASE6_V1_SUBJECTS = set(
    phase6_v1_manifest["subject"]
    .str.strip()
    .str.lower()
)

EXCLUDED_SUBJECTS = (
    PHASE5_SUBJECTS
    | PHASE6_V1_SUBJECTS
)

print(
    "Phase-5 subjects:",
    len(PHASE5_SUBJECTS)
)

print(
    "Phase-6 v1 subjects:",
    len(PHASE6_V1_SUBJECTS)
)

print(
    "Total excluded subjects:",
    len(EXCLUDED_SUBJECTS)
)

Phase-5 subjects: 64
Phase-6 v1 subjects: 128
Total excluded subjects: 186


In [44]:
rng2 = np.random.default_rng(
    20260910
)

In [45]:
fresh_cases_v2 = []

indices = rng2.permutation(
    len(counterfact)
)

for idx in indices:

    record = counterfact[int(idx)]

    subject_key = (
        record[
            "requested_rewrite"
        ]["subject"]
        .strip()
        .lower()
    )

    # Importantissimo:
    # niente soggetti già osservati
    if subject_key in EXCLUDED_SUBJECTS:
        continue

    case = prepare_fresh_case(
        record,
        int(idx)
    )

    if case is None:
        continue

    fresh_cases_v2.append(case)

    print(
        len(fresh_cases_v2),
        case["subject"],
        "| reference TE =",
        round(
            case["TE_reference"],
            3
        )
    )

    if len(
        fresh_cases_v2
    ) >= N_FRESH:

        break

1 Ring 2 | reference TE = 1.504
2 Portobello, Edinburgh | reference TE = 5.71
3 Mariano Gonzalvo | reference TE = 2.677
4 Alain de Benoist | reference TE = 2.98
5 Black Sparrow Books | reference TE = 1.306
6 Margaret Walker | reference TE = 2.006
7 Jimmy Choo | reference TE = 3.421
8 Carnival of Venice | reference TE = 9.93
9 Bavaria | reference TE = 1.14
10 Guillaumes | reference TE = 1.645
11 Johann Andreas Schmeller | reference TE = 5.039
12 Obsidian Entertainment | reference TE = 4.125
13 Il Gazzettino | reference TE = 2.327
14 Alain Mabanckou | reference TE = 4.218
15 Jules Paul Benjamin Delessert | reference TE = 1.274
16 Stevie Wonder | reference TE = 2.987
17 Sous le vent | reference TE = 1.319
18 Berlin International Film Festival | reference TE = 9.268
19 Dmitri Nabokov | reference TE = 1.016
20 Claudio Abbado | reference TE = 3.329
21 Acura Legend | reference TE = 4.726
22 Jean-Michel Jarre | reference TE = 1.586
23 Heilmann & Littmann | reference TE = 3.068
24 Autonomous Un

In [50]:
assert len(
    fresh_cases_v2
) == N_FRESH

new_subjects = {
    c["subject"]
    .strip()
    .lower()
    for c in fresh_cases_v2
}

assert (
    new_subjects
    & EXCLUDED_SUBJECTS
) == set()

print(
    "Fresh v2 subjects:",
    len(new_subjects)
)

print(
    "Overlap with previous work:",
    len(
        new_subjects
        & EXCLUDED_SUBJECTS
    )
)

Fresh v2 subjects: 127
Overlap with previous work: 0


In [51]:
MANIFEST_V2 = (
    "phase6b_locked_manifest.csv"
)

RESULT_PATH = (
    "phase6b_locked_results.csv"
)

PROTOCOL_V2 = (
    "phase6b_locked_protocol.json"
)

In [48]:
fresh_manifest_v2 = pd.DataFrame([
    {
        "case_id":
            c["case_id"],

        "dataset_index":
            c["dataset_index"],

        "subject":
            c["subject"],

        "prompt":
            c["prompt"],

        "true_text":
            c["true_text"],

        "new_text":
            c["new_text"],

        "D_clean":
            c["D_clean"],

        "TE_reference":
            c["TE_reference"]
    }

    for c in fresh_cases_v2
])

fresh_manifest_v2.to_csv(
    MANIFEST_V2,
    index=False
)

In [49]:
protocol_v2 = {
    "experiment":
        "Phase 6B locked replication",

    "model":
        MODEL_ID,

    "N_fresh":
        128,

    "excluded_phase5_subjects":
        len(PHASE5_SUBJECTS),

    "excluded_phase6_v1_subjects":
        len(PHASE6_V1_SUBJECTS),

    "subject_disjoint":
        True,

    "AIE_pair":
        AIE_PAIR,

    "interaction_pair":
        INTERACTION_PAIR,

    "primary_scale":
        PRIMARY_SCALE,

    "noise_scales":
        NOISE_SCALES,

    "noise_seeds":
        NOISE_SEEDS,

    "primary_metric":
        "normalized causal recovery",

    "primary_success":
        "paired bootstrap 95% CI lower bound > 0",

    "practical_threshold":
        0.02
}

In [ ]:
# Nuovo fix senza overlap
rng = np.random.default_rng(
    20260909
)

indices = rng.permutation(
    len(counterfact)
)

fresh_cases = []

for idx in indices:

    record = counterfact[
        int(idx)
    ]

    case_id = int(
        record["case_id"]
    )

    subject_key = (
        record["requested_rewrite"]["subject"]
        .strip()
        .lower()
    )

    if case_id in OLD_CASE_IDS:
        continue

    case = prepare_fresh_case(
        record,
        int(idx)
    )

    if case is None:
        continue

    if subject_key in OLD_SUBJECTS:
        continue

    fresh_cases.append(case)

    print(
        len(fresh_cases),
        case["subject"],
        "| reference TE =",
        round(
            case["TE_reference"],
            3
        )
    )

    if len(fresh_cases) >= N_FRESH:
        break

In [14]:
print(
    "fresh usable facts:",
    len(fresh_cases)
)

assert len(fresh_cases) == N_FRESH

assert not (
    set(
        c["case_id"]
        for c in fresh_cases
    )
    &
    OLD_CASE_IDS
)

fresh usable facts: 128


In [15]:
fresh_manifest = pd.DataFrame([
    {
        "case_id":
            c["case_id"],

        "dataset_index":
            c["dataset_index"],

        "subject":
            c["subject"],

        "prompt":
            c["prompt"],

        "true_text":
            c["true_text"],

        "new_text":
            c["new_text"],

        "D_clean":
            c["D_clean"],

        "TE_reference":
            c["TE_reference"]
    }
    for c in fresh_cases
])

fresh_manifest.to_csv(
    "phase6_fresh_manifest.csv",
    index=False
)

fresh_manifest.head()

,case_id,dataset_index,subject,prompt,true_text,new_text,D_clean,TE_reference
0,2683,675,thorium,"thorium, which is called after",Thor,Tampa,8.349422,3.971287
1,12464,628,Co-operatives UK,"Co-operatives UK, that originated in",Manchester,Paris,1.105858,3.127007
2,14427,1084,Uruguayan Football Association,Uruguayan Football Association is a part of the,FIFA,Avengers,8.934448,9.509460
3,8237,810,Ferrari 250 GTO,"Ferrari 250 GTO, developed by",Ferrari,Toyota,2.133591,3.757744
4,13213,2142,Kevin Garnett,Kevin Garnett is a professional,basketball,baseball,1.417137,1.641838


In [16]:
protocol = {
    "model": MODEL_ID,

    "N_fresh": N_FRESH,

    "AIE_pair":
        AIE_PAIR,

    "interaction_pair":
        INTERACTION_PAIR,

    "primary_scale":
        PRIMARY_SCALE,

    "noise_scales":
        NOISE_SCALES,

    "noise_seeds":
        NOISE_SEEDS,

    "reference_scale":
        REFERENCE_SCALE,

    "reference_seed":
        REFERENCE_SEED,

    "min_clean_margin":
        MIN_CLEAN_MARGIN,

    "min_reference_TE":
        MIN_REFERENCE_TE,

    "min_eval_TE":
        MIN_EVAL_TE,

    "primary_metric":
        "normalized causal recovery",

    "primary_success":
        "paired bootstrap 95% CI lower bound > 0",

    "practical_threshold":
        0.02
}

with open(
    "phase6_frozen_protocol.json",
    "w"
) as f:

    json.dump(
        protocol,
        f,
        indent=2
    )

In [17]:
def patched_margin(
    case,
    clean_states,
    corrupt_emb,
    sites
):

    patches_by_layer = {}

    for layer, role in sites:

        patches_by_layer.setdefault(
            layer,
            []
        ).append(role)

    handles = []

    for layer, roles in (
        patches_by_layer.items()
    ):

        def make_hook(
            layer_idx,
            roles_here
        ):

            def hook(
                module,
                inputs,
                output
            ):

                if isinstance(
                    output,
                    tuple
                ):

                    hidden = (
                        output[0]
                        .clone()
                    )

                else:

                    hidden = (
                        output.clone()
                    )

                for role in roles_here:

                    if role == "S":

                        pos = (
                            case[
                                "subject_last"
                            ]
                        )

                    elif role == "Q":

                        pos = (
                            case[
                                "query_pos"
                            ]
                        )

                    else:

                        raise ValueError(
                            role
                        )

                    clean_value = (
                        clean_states[
                            layer_idx + 1
                        ][:, pos, :]
                    )

                    hidden[
                        :, pos, :
                    ] = clean_value

                if isinstance(
                    output,
                    tuple
                ):

                    return (
                        hidden,
                        *output[1:]
                    )

                return hidden

            return hook

        h = (
            model.transformer.h[
                layer
            ]
            .register_forward_hook(
                make_hook(
                    layer,
                    roles
                )
            )
        )

        handles.append(h)

    try:

        with torch.no_grad():

            out = model(
                inputs_embeds=
                    corrupt_emb,

                attention_mask=
                    case[
                        "attention_mask"
                    ]
            )

        return logit_margin(
            out.logits,
            case["true_id"],
            case["new_id"]
        )

    finally:

        for h in handles:
            h.remove()

In [18]:
def evaluate_case_condition(
    case,
    noise_multiplier,
    noise_seed
):

    with torch.no_grad():

        clean_out = model(
            input_ids=
                case["input_ids"],

            attention_mask=
                case[
                    "attention_mask"
                ],

            output_hidden_states=True
        )

    clean_states = (
        clean_out.hidden_states
    )

    D_clean = logit_margin(
        clean_out.logits,
        case["true_id"],
        case["new_id"]
    )

    actual_seed = (
        int(noise_seed)
        +
        int(case["case_id"])
    )

    corrupt_emb = (
        corrupted_embeddings(
            case["input_ids"],
            case[
                "subject_positions"
            ],
            actual_seed,
            noise_multiplier
        )
    )

    with torch.no_grad():

        corrupt_out = model(
            inputs_embeds=
                corrupt_emb,

            attention_mask=
                case[
                    "attention_mask"
                ]
        )

    D_corr = logit_margin(
        corrupt_out.logits,
        case["true_id"],
        case["new_id"]
    )

    TE = D_clean - D_corr

    D_AIE = patched_margin(
        case,
        clean_states,
        corrupt_emb,
        AIE_PAIR
    )

    D_INT = patched_margin(
        case,
        clean_states,
        corrupt_emb,
        INTERACTION_PAIR
    )

    valid = (
        TE >= MIN_EVAL_TE
    )

    if valid:

        rec_AIE = (
            D_AIE - D_corr
        ) / TE

        rec_INT = (
            D_INT - D_corr
        ) / TE

        delta_rec = (
            rec_INT - rec_AIE
        )

    else:

        rec_AIE = np.nan
        rec_INT = np.nan
        delta_rec = np.nan

    return {
        "case_id":
            case["case_id"],

        "noise_scale":
            noise_multiplier,

        "noise_seed":
            noise_seed,

        "D_clean":
            D_clean,

        "D_corr":
            D_corr,

        "TE":
            TE,

        "valid":
            valid,

        "D_AIE":
            D_AIE,

        "D_INT":
            D_INT,

        "recovery_AIE":
            rec_AIE,

        "recovery_INT":
            rec_INT,

        "delta_recovery":
            delta_rec,

        "correct_AIE":
            float(D_AIE > 0),

        "correct_INT":
            float(D_INT > 0),

        "margin_advantage_INT":
            D_INT - D_AIE
    }

In [19]:
RESULT_PATH = (
    "phase6_validation_results.csv"
)

if os.path.exists(
    RESULT_PATH
):

    phase6_results = (
        pd.read_csv(
            RESULT_PATH
        )
    )

else:

    phase6_results = (
        pd.DataFrame()
    )

In [20]:
done = set()

if len(phase6_results) > 0:

    done = set(
        zip(
            phase6_results[
                "case_id"
            ],

            phase6_results[
                "noise_scale"
            ],

            phase6_results[
                "noise_seed"
            ]
        )
    )

In [21]:
rows = (
    phase6_results
    .to_dict("records")
    if len(phase6_results)
    else []
)

for ci, case in enumerate(
    fresh_cases
):

    for scale in NOISE_SCALES:

        for seed in NOISE_SEEDS:

            key = (
                case["case_id"],
                scale,
                seed
            )

            if key in done:
                continue

            row = (
                evaluate_case_condition(
                    case,
                    scale,
                    seed
                )
            )

            rows.append(row)

            pd.DataFrame(
                rows
            ).to_csv(
                RESULT_PATH,
                index=False
            )

    print(
        f"{ci+1}/{len(fresh_cases)}",
        case["subject"]
    )

phase6_results = pd.DataFrame(
    rows
)

1/128 thorium
2/128 Co-operatives UK
3/128 Uruguayan Football Association
4/128 Ferrari 250 GTO
5/128 Kevin Garnett
6/128 Mauritius
7/128 Il giuramento
8/128 Victoria Derbyshire
9/128 Orpheus Island National Park
10/128 Il Posto
11/128 Aleksander Griboyedov
12/128 Everberg
13/128 Ghora Gali
14/128 Honda CRF series
15/128 Andrei Bitov
16/128 2004 Australian Open
17/128 Ludovic Sylvestre
18/128 Korean Wikipedia
19/128 Babylas of Antioch
20/128 North Dakota
21/128 Chicago Rockets
22/128 Veronica Lario
23/128 Pulaski County Special School District
24/128 Great Soviet Encyclopedia
25/128 Seattle International Film Festival
26/128 La Gomera
27/128 Kaiketsu Lion-Maru
28/128 Gammalsvenskby
29/128 Smith Glacier
30/128 Ely Guerra
31/128 Logan Thomas
32/128 Jeremy Allen White
33/128 Carl Mayer
34/128 Sangamam
35/128 Francesco Furini
36/128 visual kei
37/128 Ferrari Berlinetta Boxer
38/128 Accademia Pontaniana
39/128 Saint-Domingue expedition
40/128 Beijing College Student Film Festival
41/128 Mid

In [43]:
# Prima diagnostica
print(
    phase6_results[
        "valid"
    ].value_counts()
)

print()

print(
    phase6_results.groupby(
        "noise_scale"
    )[
        [
            "TE",
            "recovery_AIE",
            "recovery_INT",
            "delta_recovery"
        ]
    ].mean()
)

valid
True     1105
False      47
Name: count, dtype: int64

                   TE  recovery_AIE  recovery_INT  delta_recovery
noise_scale                                                      
2.5          3.077314      0.797348      0.847616        0.050268
3.0          3.189955      0.790978      0.838639        0.047661
3.5          3.237234      0.788599      0.835428        0.046829


In [23]:
primary = (
    phase6_results[
        (
            phase6_results[
                "noise_scale"
            ]
            == PRIMARY_SCALE
        )
        &
        (
            phase6_results[
                "valid"
            ]
        )
    ]
)

In [25]:
primary_case = (
    primary
    .groupby("case_id")
    .agg(
        recovery_AIE=(
            "recovery_AIE",
            "mean"
        ),

        recovery_INT=(
            "recovery_INT",
            "mean"
        ),

        delta=(
            "delta_recovery",
            "mean"
        ),

        correct_AIE=(
            "correct_AIE",
            "mean"
        ),

        correct_INT=(
            "correct_INT",
            "mean"
        )
    )
)

In [26]:
print(
    primary_case.describe()
)

print(
    "\nMean AIE recovery:",
    primary_case[
        "recovery_AIE"
    ].mean()
)

print(
    "Mean interaction recovery:",
    primary_case[
        "recovery_INT"
    ].mean()
)

print(
    "Mean delta:",
    primary_case[
        "delta"
    ].mean()
)

       recovery_AIE  recovery_INT       delta  correct_AIE  correct_INT
count    128.000000    128.000000  128.000000   128.000000        128.0
mean       0.794510      0.843226    0.048716     0.989583          1.0
std        0.141963      0.133094    0.069854     0.058225          0.0
min        0.346534      0.371348   -0.213113     0.666667          1.0
25%        0.720842      0.778322    0.016452     1.000000          1.0
50%        0.811868      0.851943    0.030021     1.000000          1.0
75%        0.877938      0.923324    0.063474     1.000000          1.0
max        1.146169      1.133509    0.404616     1.000000          1.0

Mean AIE recovery: 0.7945099801125834
Mean interaction recovery: 0.8432262714794663
Mean delta: 0.04871629136688285


In [29]:
def bootstrap_mean_ci(
    values,
    n_boot=10000,
    seed=12345
):

    x = np.asarray(
        values,
        dtype=float
    )

    rng = np.random.default_rng(
        seed
    )

    boot = np.empty(
        n_boot
    )

    n = len(x)

    for b in range(n_boot):

        sample = rng.choice(
            x,
            size=n,
            replace=True
        )

        boot[b] = (
            sample.mean()
        )

    return (
        x.mean(),
        np.quantile(
            boot,
            0.025
        ),
        np.quantile(
            boot,
            0.975
        )
    )

In [30]:
delta_mean, ci_lo, ci_hi = (
    bootstrap_mean_ci(
        primary_case[
            "delta"
        ]
    )
)

print(
    "PRIMARY RESULT"
)

print(
    "delta mean:",
    delta_mean
)

print(
    "95% CI:",
    ci_lo,
    ci_hi
)

print(
    "statistical success:",
    ci_lo > 0
)

print(
    "practical success:",
    delta_mean >= 0.02
)

PRIMARY RESULT
delta mean: 0.04871629136688285
95% CI: 0.03731224174649952 0.06128446478952096
statistical success: True
practical success: True


In [32]:
# F6B
robust_rows = []

for scale in NOISE_SCALES:

    d = (
        phase6_results[
            (
                phase6_results[
                    "noise_scale"
                ] == scale
            )
            &
            phase6_results[
                "valid"
            ]
        ]
    )

    by_case = (
        d.groupby(
            "case_id"
        )[
            "delta_recovery"
        ]
        .mean()
    )

    mean_d, lo, hi = (
        bootstrap_mean_ci(
            by_case.values
        )
    )

    robust_rows.append({
        "scale":
            scale,

        "n_cases":
            len(by_case),

        "delta_mean":
            mean_d,

        "CI_lo":
            lo,

        "CI_hi":
            hi,

        "positive":
            lo > 0
    })

robustness = pd.DataFrame(
    robust_rows
)

print(robustness)

   scale  n_cases  delta_mean     CI_lo     CI_hi  positive
0    2.5      128    0.050677  0.039568  0.063057      True
1    3.0      128    0.048716  0.037312  0.061284      True
2    3.5      128    0.049046  0.038307  0.061176      True


In [33]:
win_rate = (
    primary_case[
        "delta"
    ] > 0
).mean()

tie_rate = (
    np.isclose(
        primary_case[
            "delta"
        ],
        0
    )
).mean()

print(
    "interaction win rate:",
    win_rate
)

print(
    "tie rate:",
    tie_rate
)

interaction win rate: 0.9140625
tie rate: 0.0
